# Lekcja 05 - Agentic RAG


## Setup

This notebook demonstrates the Agentic RAG (Retrieval-Augmented Generation) pattern using the Microsoft Agent Framework.

**Prerequisites:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — your Azure AI Search service endpoint
- `AZURE_SEARCH_API_KEY` — your Azure AI Search API key
- Azure OpenAI deployment configured via environment variables
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Czym jest Agentic RAG?

Tradycyjny RAG działa według ustalonego schematu: najpierw pobiera dokumenty, a następnie generuje odpowiedź. **Agentic RAG** idzie o krok dalej, dając agentowi autonomię w decydowaniu, **kiedy** i **jak** pobierać informacje.

Dzięki Agentic RAG agent może:
- **Zdecydować**, czy potrzebne jest pobranie informacji przed udzieleniem odpowiedzi na pytanie
- **Wybrać**, które źródło danych lub narzędzie zapytać
- **Ocenić** uzyskane wyniki i wykonać dalsze pobrania, jeśli pierwsza próba jest niewystarczająca
- **Połączyć** informacje z wielu etapów pobierania w spójną odpowiedź

To sprawia, że agent jest bardziej elastyczny i precyzyjny w porównaniu do statycznego schematu pobierz-then-generate.


## Tworzenie narzędzia do wyszukiwania

W Agentic RAG zewnętrzne źródła danych są opakowane jako **narzędzia**, które agent może wywołać na żądanie. Pozwala to agentowi traktować wyszukiwanie jako kolejną akcję, którą może podjąć, a nie jako obowiązkowy krok.

Poniżej definiujemy bazę wiedzy o podróżach i udostępniamy ją jako narzędzie, które agent może wywołać, aby wyszukać informacje o destynacji.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Budowanie agenta RAG

Teraz tworzymy agenta, który jest instruowany, aby **zawsze najpierw pobierał informacje zanim odpowie**. Agent używa narzędzia `search_travel_knowledge`, aby oprzeć swoje odpowiedzi na bazie wiedzy, zamiast polegać na własnych danych treningowych.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iteracyjne wyszukiwanie — wzorzec Maker-Checker

Kluczową zaletą Agentic RAG jest **iteracyjne wyszukiwanie**. Agent może przeprowadzać wielokrotne rundy wyszukiwania, aby zweryfikować, poprawić lub rozszerzyć swoje początkowe ustalenia — podobnie jak w procesie "maker-checker":

1. **Krok maker**: Agent pobiera początkowe informacje i tworzy szkic odpowiedzi.
2. **Krok checker**: Agent wykonuje dodatkowe wyszukiwania, aby zweryfikować szczegóły lub uzupełnić luki.

Poniżej agentowi zadano pytanie, które wymaga porównania kilku miejsc docelowych, co skłania go do wielokrotnego wyszukiwania.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Podsumowanie

W tej lekcji nauczyłeś się, jak zbudować system **Agentic RAG** korzystając z Microsoft Agent Framework:

- **Agentic RAG** pozwala agentom autonomicznie decydować, kiedy pobierać informacje, co sprawia, że pobieranie jest dynamiczne, a nie stałe.
- **Narzędzia jako źródła danych**: Zewnętrzne bazy wiedzy (takie jak Azure AI Search) są opakowane jako narzędzia, które agent może wywoływać.
- **Iteracyjne pobieranie**: Wzorzec maker-checker umożliwia agentowi wykonanie wielu rund pobierania — wyszukiwania, weryfikacji i usprawniania — zanim zostanie wygenerowana ostateczna odpowiedź.

W produkcji zastąpiłbyś pamięciową `TRAVEL_KNOWLEDGE_BASE` rzeczywistym indeksem Azure AI Search do obsługi masowego pobierania dokumentów podróżnych.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
